**Objectif du projet: Pr?diction du CA (Sales_Amount)**

Phase 1  Collecte et exploration

In [ ]:
!pip install -U gradio

In [ ]:
import gradio as gr
gr.__version__

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,  GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score
RANDOM_STATE = 42
import joblib
import json
import joblib
import pandas as pd
import gradio as gr


In [ ]:
df =pd.read_csv("Fashion_sales.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()
print(df.describe(include="all").T)

# V?rification variable cible
print(df["Profit"].describe())

Phase 2: Visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["Profit"], bins=40, kde=True, ax=ax[0])
ax[0].set_title("Distribution de Profit")

sns.boxplot(x=df["Profit"], ax=ax[1])
ax[1].set_title("Boxplot de Profit")

plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=df, x="Units_Sold", y="Profit", alpha=0.5)
plt.title("Units_Sold vs Profit")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Segment", y="Profit")
plt.title("Profit par Segment")
plt.show()

**Phase 3 ? Nettoyage & pr?paration**

In [ ]:
Valeurs_manquantes = df.isna().sum().sort_values(ascending=False)
print(Valeurs_manquantes[Valeurs_manquantes > 0])

print("Doublons:", df.duplicated().sum())

In [ ]:
target = "Profit"

numeric_features = ["Units_Sold", "Unit_Price", "Discount_%"]
categorical_features = ["Segment", "Product_Category", "City"]

features = numeric_features + categorical_features

In [ ]:
target="Profit"
features = ["Units_Sold", "Unit_Price", "Discount_%", "Segment", "Product_Category", "City"]
df_model = df[features + [target]].copy()

In [ ]:
target = "Profit"
features = ["Units_Sold","Unit_Price","Discount_%","Segment","Product_Category","City"]

X = df_model[features]
y = df_model[target]

X.shape, y.shape

? X contient les 6 les features, y = Profit ?

In [ ]:
df_model = df_model[df_model["Units_Sold"].isna() | (df_model["Units_Sold"] >= 0)].copy()

In [ ]:
# 1) Combien de valeurs n?gatives ?
(df_model["Units_Sold"] < 0).sum()

In [ ]:
# Traitement des r?duction > 100%
df_model["Discount_%"] = df_model["Discount_%"].clip(0, 1)

In [ ]:
# V?rification> 1
(df_model["Discount_%"] > 1).sum(), df_model["Discount_%"].max()

In [ ]:
target = "Profit"
features = ["Units_Sold", "Unit_Price", "Discount_%", "Segment", "Product_Category", "City"]

df_model = df[features + [target]].copy()

df_model.isna().sum().sort_values(ascending=False)

In [ ]:
# Identification des outliers de notre valeur variable cible par m?thode IQR
Q1 = df_model["Profit"].quantile(0.25)
Q3 = df_model["Profit"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

Q1, Q3, IQR, lower_bound, upper_bound


In [ ]:
outliers_mask = (df_model["Profit"] < lower_bound) | (df_model["Profit"] > upper_bound)
outliers_count = outliers_mask.sum()
outliers_pct = outliers_count / len(df_model) * 100

outliers_count, outliers_pct

In [ ]:
# Traitement des outliers par la m?thode de winsorisation
df_model_iqr = df_model.copy()
df_model_iqr["Profit"] = df_model_iqr["Profit"].clip(lower_bound, upper_bound)

df_model_iqr["Profit"].describe()

- Les profits trop faibles sont r?duits ?lower_bound
- Les profits trop ?lev?s sont r?duits ?upper_bound
- On garde toutes les lignes (pas de perte de donn?es)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,4))

sns.boxplot(x=df_model["Profit"], ax=ax[0])
ax[0].set_title("Avant traitement outliers (Profit)")

sns.boxplot(x=df_model_iqr["Profit"], ax=ax[1])
ax[1].set_title("Apr?s cap IQR (Profit)")

plt.tight_layout()
plt.show()

**Phase 4 ? Normalisation & s?paration**

In [ ]:
#S?paration des donn?es d'entra?nement et de test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
#D?finition des colonnes num?rique et cat?gorielles
target = "Profit"

numeric_features = ["Units_Sold", "Unit_Price", "Discount_%"]
categorical_features = ["Segment", "Product_Category", "City"]

features = numeric_features + categorical_features

In [ ]:
# le pr?processeur (imputation + mise ? l'?chelle + encodage)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Pr?processeur (Pipeline sklearn)
On construit un pr?processeur unique avec ColumnTransformer afin d'appliquer automatiquement les transformations adapt?es ? chaque type de variable.

Pour les variables num?riques ( Units_Sold, Unit_Price, Discount_%) : on impute les valeurs manquantes avec la m?diane, puis on standardise avec StandardScalerpour mettre toutes les caract?ristiques sur une ?chelle comparable.
Pour les variables cat?gorielles ( Segment, Product_Category, City) : on impute les valeurs manquantes avec la valeur la plus fr?quente, puis on encode en One-Hot via OneHotEncoder(handle_unknown="ignore")afin de transformer les cat?gories en variables num?riques
Ce choix garantit un pipeline reproductible , ?vitant la fuite de donn?es et est directement r?utilisable pour le d?ploiement sur (Gradio/Hugging Face).

In [ ]:
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape :", X_test_processed.shape)

**Phase 5 ? Mod?lisation & ?valuation**

In [ ]:
# On Cr?e une fonction d'?valuation

def train_eval(name, model):
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)

    return {"model": name, "rmse": rmse, "r2": r2, "pipeline": pipe}

results = []
results.append(train_eval("LinearRegression", LinearRegression()))
results.append(train_eval("RandomForest", RandomForestRegressor(
    n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1
)))
results.append(train_eval("GradientBoosting", GradientBoostingRegressor(
    random_state=RANDOM_STATE
)))

df_results = pd.DataFrame([{k:v for k,v in r.items() if k!="pipeline"} for r in results]).sort_values("rmse")
df_results

On scale tout dans une Pipeline:
le preprocessorg?rer valeurs manquantes + normalisation + one-hot
puis le mod?le apprend sur les donn?es transform?es
On ?value sur X_testpour mesurer la g?n?ralisation.
On retourne ?galement le pipesuivre (pratique pour sauvegarder le meilleur ensuite)

In [ ]:
#Entra?nement de 2 mod?les
results = []

# Mod?le 1 : R?gression lin?aire
results.append(train_eval(
    "LinearRegression",
    LinearRegression()
))

In [ ]:
# Mod?le 2 : Random Forest
results.append(train_eval(
    "RandomForestRegressor",
    RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
))

In [ ]:
df_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != "pipeline"}
    for r in results
])

df_results = df_results.sort_values("rmse")
df_results

In [ ]:
best = min(results, key=lambda d: d["rmse"])
best_name = best["model"]
best_pipe = best["pipeline"]

print("Best model:", best_name)
display(df_results.sort_values("rmse"))

joblib.dump(best_pipe, "model.joblib")
print("Saved -> model.joblib")

- Meilleur des deux : LinearRegression (RMSE plus bas et R? moins n?gatif).

M?me si les performances (R?) restent faibles, nous d?ployons le mod?le afin de valider le pipeline complet (pr?traitement, entra?nement, s?rialisation, interface Gradio). Les r?sultats attendus que Profitd?pendent probablement de variables non disponibles dans le dataset (co?ts/marges), ce qui limite le pouvoir pr?dictif.

**Phase 6 ? D?ploiement Gradio**

In [ ]:
# Sauvegarde de pipline
joblib.dump(best_pipe, "model.joblib")

In [ ]:
import json

meta = {
    "segment_choices": sorted(df_model["Segment"].dropna().unique().tolist()),
    "category_choices": sorted(df_model["Product_Category"].dropna().unique().tolist()),
    "city_choices": sorted(df_model["City"].dropna().unique().tolist()),
}

with open("meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [ ]:
#  Cr?ation app.py

# Charger le pipeline (preprocessing + mod?le)
model = joblib.load("model.joblib")

# Charger les valeurs possibles pour les dropdowns
with open("meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

segment_choices = meta["segment_choices"]
category_choices = meta["category_choices"]
city_choices = meta["city_choices"]


def predict_profit(units_sold, unit_price, discount, segment, product_category, city):
    """
    Re?oit les inputs utilisateur (Gradio),
    construit un DataFrame avec les m?mes colonnes que pendant l'entra?nement,
    puis renvoie la pr?diction du Profit.
    """
    # Construire une ligne d'entr?e sous forme DataFrame
    X = pd.DataFrame([{
        "Units_Sold": units_sold,
        "Unit_Price": unit_price,
        "Discount_%": discount,
        "Segment": segment,
        "Product_Category": product_category,
        "City": city
    }])

    pred = model.predict(X)[0]
    return float(pred)


title = "Pr?diction du Profit (R?gression Lin?aire)"
description = (
    "Ce Space pr?dit le **Profit** ? partir des caract?ristiques d'une commande. "
    "Mod?le : **Linear Regression** (baseline)."
)

with gr.Blocks(theme=gr.themes.Soft(), title=title) as demo:
    gr.Markdown(f"# {title}\n{description}")

    with gr.Row():
        units_sold = gr.Number(label="Units_Sold (nombre d'unit?s vendues)", value=1)
        unit_price = gr.Number(label="Unit_Price (prix unitaire)", value=1000)

    with gr.Row():
        discount = gr.Number(label="Discount_% (ex: 0.2 = 20%)", value=0.0)

    with gr.Row():
        segment = gr.Dropdown(segment_choices, label="Segment")
        product_category = gr.Dropdown(category_choices, label="Product_Category")
        city = gr.Dropdown(city_choices, label="City")

    btn = gr.Button("Pr?dire le Profit")
    output = gr.Number(label="Profit pr?dit")

    btn.click(
        fn=predict_profit,
        inputs=[units_sold, unit_price, discount, segment, product_category, city],
        outputs=output
    )

demo.launch()